# Generate ProSE Protein Embeddings

**Environment:** bioemb

**Purpose:** Generate average-pooled ProSE embeddings for BindingDB protein sequences. These embeddings condition the CTDDG model during fine-tuning.

**Workflow:**

1. Configure repository and dataset paths.
2. Check the environment, ProSE model weights, and GPU availability.
3. Identify missing or incomplete embeddings.
4. Generate embeddings in resumable shards.
5. Validate the generated HDF5 files.
6. Prepare a Slurm job for execution on Bhavani.

**Output:** One 6,165-dimensional embedding per protein, stored in an HDF5 file.

## Import libraries

In [1]:
import os
import sys
import glob
import json
import time
import shutil
import inspect
import subprocess
from datetime import timedelta

import numpy as np

## 1. Configuration

Define the repository paths, dataset location, ProSE model location, and embedding parameters. The notebook uses these settings throughout the workflow, so paths should be defined in one place.

For the initial experiment, we will process only fold `d1`, including its training and test splits. The other folds can be enabled later.

In [2]:
REPO_ROOT = os.environ.get(
    "CTDDG_REPO_ROOT",
    "/home/muhamed/repo/CTDDG"
)

BINDINGDB_DIR = os.path.join(
    REPO_ROOT, "datasets/data/bindingdb"
)

SCRIPTS_DIR = os.path.join(REPO_ROOT, "scripts")
CLUSTER_DIR = os.path.join(REPO_ROOT, "cluster")
LOGS_DIR = os.path.join(REPO_ROOT, "logs")

# ProSE checkout containing embed_sequences.py and saved_models/
"""
PROSE_DIR = os.environ.get(
    "CTDDG_PROSE_DIR",
    os.path.join(REPO_ROOT, "prose")
)
"""
PROSE_DIR = "/home/muhamed/repo/CTDDG/prose"
# Embedding configuration
N_C = 6165
SHARD_SIZE = 128
DEVICE = -1
KEEP_SHARDS = False

# BindingDB fold directories
FOLD_SUFFIX = {
    "d1": "4_org_1042_104",
    "d2": "4_org_1000_112",
    "d3": "4_org_1004_122",
    "d4": "4_org_1002_103",
    "d5": "4_org_1036_124",
}

FOLDS = ["d1"]
SPLITS = ["train", "test"]

for directory in (SCRIPTS_DIR, CLUSTER_DIR, LOGS_DIR):
    os.makedirs(directory, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"BindingDB:  {BINDINGDB_DIR}")
print(f"ProSE:      {PROSE_DIR}")
print(f"Folds:      {FOLDS}")
print(f"Splits:     {SPLITS}")

Repository: /home/muhamed/repo/CTDDG
BindingDB:  /home/muhamed/repo/CTDDG/datasets/data/bindingdb
ProSE:      /home/muhamed/repo/CTDDG/prose
Folds:      ['d1']
Splits:     ['train', 'test']


## 2. Locate ProSE

Find the ProSE embedding script and verify that its pretrained model weights exist. The embedding script must be located before any protein sequences can be processed.

The model weights are already present in your repository, so this cell should resolve to the existing `prose` directory.

In [3]:
def find_prose_dir(explicit=None):
    """Locate a ProSE checkout containing embed_sequences.py."""
    candidates = []
    if explicit:
        candidates.append(explicit)
    candidates += [
        os.path.join(REPO_ROOT, "prose"),
        os.path.join(os.path.dirname(REPO_ROOT), "prose"),
        os.path.expanduser("~/prose"),
        os.path.expanduser("~/repo/prose"),
        os.path.expanduser("~/tools/prose"),
    ]
    try:
        import prose

        prose_file = getattr(prose, "__file__", None)
        if prose_file:
            candidates.append(
                os.path.dirname(os.path.dirname(os.path.abspath(prose_file)))
            )
    except ImportError:
        pass

    for c in candidates:
        if c and os.path.isfile(os.path.join(c, "embed_sequences.py")):
            return os.path.abspath(c)
    return None


report = []
ok = True

report.append(("python", sys.version.split()[0]))
report.append(("conda env", os.environ.get("CONDA_DEFAULT_ENV", "(none)")))
report.append(("sys.prefix", sys.prefix))

if os.environ.get("CONDA_DEFAULT_ENV") not in ("bioemb", None):
    print(f"[WARN] CONDA_DEFAULT_ENV is {os.environ.get('CONDA_DEFAULT_ENV')!r}, not 'bioemb'.")

try:
    import torch
    report.append(("torch", torch.__version__))
    report.append(("cuda available", str(torch.cuda.is_available())))
    if torch.cuda.is_available():
        report.append(("gpu", torch.cuda.get_device_name(0)))
except ImportError:
    report.append(("torch", "MISSING")); ok = False

try:
    import h5py
    report.append(("h5py", h5py.__version__))
except ImportError:
    report.append(("h5py", "MISSING")); ok = False

PROSE_DIR = find_prose_dir(PROSE_DIR)
if PROSE_DIR:
    report.append(("prose dir", PROSE_DIR))
    EMBED_SCRIPT = os.path.join(PROSE_DIR, "embed_sequences.py")
    weights = os.path.join(PROSE_DIR, "saved_models", "prose_mt_3x1024.sav")
    if os.path.isfile(weights):
        report.append(("prose weights", f"{os.path.getsize(weights) / 1e6:.0f} MB"))
    else:
        report.append(("prose weights", "MISSING")); ok = False
else:
    EMBED_SCRIPT = None
    report.append(("prose dir", "NOT FOUND")); ok = False

report.append(("slurm job", os.environ.get("SLURM_JOB_ID", "(not in an allocation)")))

width = max(len(k) for k, _ in report)
for k, v in report:
    print(f"  {k:<{width}}  {v}")

if not ok:
    print("\n[BLOCKED]")
    if not PROSE_DIR:
        print("  ProSE not found. Clone it and point CTDDG_PROSE_DIR at the checkout:")
        print("    git clone https://github.com/tbepler/prose.git ~/prose")
    print("  Pre-trained weights (compute nodes usually have no internet, so fetch on the "
          "login node):")
    print("    cd <prose_dir> && wget http://bergerlab-downloads.csail.mit.edu/prose/"
          "saved_models.zip && unzip saved_models.zip")
else:
    print("\n[OK] ready to embed.")

/home/muhamed/.conda/envs/bioemb/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  python          3.9.25
  conda env       bioemb
  sys.prefix      /home/muhamed/.conda/envs/bioemb
  torch           1.10.0+cu102
  cuda available  True
  gpu             NVIDIA L40
  h5py            3.14.0
  prose dir       /home/muhamed/repo/CTDDG/prose
  prose weights   391 MB
  slurm job       12831

[OK] ready to embed.


/home/muhamed/.conda/envs/bioemb/lib/python3.9/site-packages/torch/cuda/__init__.py:143: UserWarning: 
NVIDIA L40 with CUDA capability sm_89 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70.
If you want to use the NVIDIA L40 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


## 3. Inspect Existing Embeddings

Inspect the BindingDB protein FASTA files and determine which embedding files are already complete, missing, partially generated, or invalid.

The validation checks that each protein has a corresponding embedding with the expected 6,165-dimensional shape and finite values. Completed shards are also checked so interrupted embedding jobs can resume.

In [4]:
def split_paths(fold, split):
    """FASTA, protein-list and output-embedding paths for one split."""
    tag = "tr" if split == "train" else "te"
    d = os.path.join(BINDINGDB_DIR, f"{split}_dataset", f"{split}_{FOLD_SUFFIX[fold]}")
    return {
        "dir": d,
        "fasta": os.path.join(d, f"{fold}_{tag}_unique_proteins_bepler.fa"),
        "proteins": os.path.join(d, f"{fold}_{tag}_unique_proteins.txt"),
        "out": os.path.join(d, f"{fold}_{tag}_unique_embeddings_bepler.h5"),
        "work": os.path.join(d, f".prose_work_{fold}_{tag}"),
    }


def read_fasta(path):
    """[(name, sequence)] with name taken up to the first whitespace, as prose does."""
    records, name, buf = [], None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if name is not None:
                    records.append((name, ''.join(buf)))
                name, buf = line[1:].split()[0], []
            elif line:
                buf.append(line)
    if name is not None:
        records.append((name, ''.join(buf)))
    return records


def validate_h5(path, expect_names, n_c=N_C):
    """-> (n_valid, problem_or_None). Mirrors the fine-tuning loader's checks."""
    if not os.path.isfile(path):
        return 0, "absent"
    import h5py
    try:
        with h5py.File(path, 'r') as f:
            keys = set(f.keys())
            missing = [n for n in expect_names if n not in keys]
            extra = [k for k in keys if k not in set(expect_names)]
            if missing:
                return len(expect_names) - len(missing), f"{len(missing)} proteins missing"
            if extra:
                return len(expect_names), f"{len(extra)} unexpected keys"
            for n in expect_names:
                a = np.asarray(f[n]).squeeze()
                if a.ndim != 1:
                    return 0, f"{n} has shape {np.asarray(f[n]).shape} (not pooled)"
                if a.shape[0] != n_c:
                    return 0, f"{n} width {a.shape[0]} != {n_c}"
                if not np.isfinite(a).all():
                    return 0, f"{n} contains NaN/inf"
        return len(expect_names), None
    except Exception as e:
        return 0, f"unreadable ({type(e).__name__})"


def inventory():
    rows = []
    for fold in FOLDS:
        for split in SPLITS:
            p = split_paths(fold, split)
            row = {"fold": fold, "split": split, "n_proteins": 0, "embedded": 0,
                   "shards_done": 0, "status": "", "note": ""}

            if not os.path.isfile(p["fasta"]):
                row["status"] = "NO FASTA"
                row["note"] = os.path.basename(p["fasta"])
                rows.append(row)
                continue

            names = [n for n, _ in read_fasta(p["fasta"])]
            row["n_proteins"] = len(names)

            if os.path.isfile(p["proteins"]):
                n_listed = sum(1 for ln in open(p["proteins"]) if ln.strip())
                if n_listed != len(names):
                    row["note"] = f"FASTA {len(names)} vs protein list {n_listed}"

            n_valid, problem = validate_h5(p["out"], names)
            row["embedded"] = n_valid
            if problem is None:
                row["status"] = "DONE"
            elif problem == "absent":
                done = count_done_shards(p["work"], names)
                row["shards_done"] = done
                row["status"] = "PARTIAL" if done else "TODO"
            else:
                row["status"] = "REBUILD"
                row["note"] = problem
                row["shards_done"] = count_done_shards(p["work"], names)
            rows.append(row)
    return rows


def shard_plan(names, shard_size=SHARD_SIZE):
    """Deterministic shards, so resume lines up across restarts."""
    return [names[i:i + shard_size] for i in range(0, len(names), shard_size)]


def count_done_shards(work_dir, names):
    if not os.path.isdir(work_dir):
        return 0
    plan = shard_plan(names)
    return sum(1 for i, chunk in enumerate(plan)
               if validate_h5(os.path.join(work_dir, f"shard_{i:04d}.h5"), chunk)[1] is None)

In [5]:
def show_inventory(rows):
    hdr = f"{'fold':5s} {'split':6s} {'proteins':>9s} {'embedded':>9s} {'shards':>7s}  {'status':8s} note"
    print(hdr); print("-" * len(hdr))
    for r in rows:
        shards = f"{r['shards_done']}/{len(shard_plan([0] * r['n_proteins']))}" \
            if r["n_proteins"] and r["status"] in ("PARTIAL", "REBUILD") else ""
        print(f"{r['fold']:5s} {r['split']:6s} {r['n_proteins']:9d} {r['embedded']:9d} "
              f"{shards:>7s}  {r['status']:8s} {r['note']}")
    todo = [r for r in rows if r["status"] in ("TODO", "PARTIAL", "REBUILD")]
    done = [r for r in rows if r["status"] == "DONE"]
    print(f"\n{len(done)} splits complete, {len(todo)} to do, "
          f"{sum(r['n_proteins'] - r['embedded'] for r in todo)} proteins remaining")
    return todo


rows = inventory()
todo = show_inventory(rows)

fold  split   proteins  embedded  shards  status   note
-------------------------------------------------------
d1    train       1042         0          TODO     
d1    test         104         0          TODO     

0 splits complete, 2 to do, 1146 proteins remaining


## 4. Generate Embeddings in Resumable Shards

ProSE writes its output in a new HDF5 file rather than appending to an existing one. To support recovery after interruptions, each split is divided into smaller shards.

Each completed shard is validated and saved. Once all shards are complete, they are merged into the final embedding file and validated again.

In [6]:
def resolve_device(device=DEVICE):
    """embed_sequences.py: -1 = CPU, >=0 pins that GPU."""
    if device is not None:
        return int(device)
    try:
        import torch
        return 0 if torch.cuda.is_available() else -1
    except ImportError:
        return -1


def write_fasta(path, records):
    with open(path, 'w') as f:
        for name, seq in records:
            f.write(f">{name}\n{seq}\n")


def run_shard(records, shard_h5, shard_fa, device, embed_script=None, quiet=True):
    """Embed one shard with the repo's ProSE script. Returns (ok, message)."""
    embed_script = embed_script or EMBED_SCRIPT
    write_fasta(shard_fa, records)
    tmp = shard_h5 + ".tmp"
    for stale in (tmp, shard_h5):
        if os.path.exists(stale):
            os.remove(stale)

    cmd = [sys.executable, embed_script, shard_fa, "-o", tmp,
           "--pool", "avg", "-d", str(device)]
    env = dict(os.environ, PYTHONPATH=os.pathsep.join(
        [os.path.dirname(embed_script)] + ([os.environ["PYTHONPATH"]]
                                           if os.environ.get("PYTHONPATH") else [])))
    proc = subprocess.run(cmd, cwd=os.path.dirname(embed_script), env=env,
                          capture_output=True, text=True)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-6:]
        return False, "embed_sequences.py failed:\n    " + "\n    ".join(tail)

    n_valid, problem = validate_h5(tmp, [n for n, _ in records])
    if problem is not None:
        return False, f"shard output invalid: {problem}"

    os.replace(tmp, shard_h5)
    os.remove(shard_fa)
    if not quiet:
        print(f"      {n_valid} proteins")
    return True, "ok"


def merge_shards(out_path, work_dir, plan, names):
    """Concatenate validated shards into the split's embedding file, atomically."""
    import h5py
    tmp = out_path + ".tmp"
    if os.path.exists(tmp):
        os.remove(tmp)
    with h5py.File(tmp, 'w') as fo:
        for i, chunk in enumerate(plan):
            with h5py.File(os.path.join(work_dir, f"shard_{i:04d}.h5"), 'r') as fi:
                for name, _ in chunk:
                    fo.create_dataset(name, data=np.asarray(fi[name]).squeeze())
    n_valid, problem = validate_h5(tmp, names)
    if problem is not None:
        os.remove(tmp)
        raise RuntimeError(f"merged file failed validation: {problem}")
    os.replace(tmp, out_path)
    return n_valid


def embed_split(fold, split, shard_size=SHARD_SIZE, device=None, keep_shards=KEEP_SHARDS):
    """Embed one split, skipping shards already on disk. Safe to call repeatedly."""
    p = split_paths(fold, split)
    label = f"{fold}/{split}"

    if not os.path.isfile(p["fasta"]):
        print(f"[{label}] no FASTA at {p['fasta']} — skipped")
        return False

    records = read_fasta(p["fasta"])
    names = [n for n, _ in records]
    device = resolve_device() if device is None else int(device)

    if validate_h5(p["out"], names)[1] is None:
        print(f"[{label}] already complete ({len(names)} proteins) — skipped")
        return True

    os.makedirs(p["work"], exist_ok=True)
    plan = [records[i:i + shard_size] for i in range(0, len(records), shard_size)]
    print(f"[{label}] {len(names)} proteins, {len(plan)} shards of <={shard_size}, device {device}")

    t_start = time.time()
    n_run = 0
    for i, chunk in enumerate(plan):
        shard_h5 = os.path.join(p["work"], f"shard_{i:04d}.h5")
        if validate_h5(shard_h5, [n for n, _ in chunk])[1] is None:
            continue

        t0 = time.time()
        ok, msg = run_shard(chunk, shard_h5, os.path.join(p["work"], f"shard_{i:04d}.fa"),
                            device)
        if not ok:
            print(f"  shard {i + 1}/{len(plan)} FAILED — {msg}")
            print(f"  [{label}] stopping; completed shards are kept, re-run to continue.")
            return False

        n_run += 1
        dt = time.time() - t0
        remaining = sum(1 for j, c in enumerate(plan) if j > i
                        and validate_h5(os.path.join(p["work"], f"shard_{j:04d}.h5"),
                                        [n for n, _ in c])[1] is not None)
        eta = timedelta(seconds=int((time.time() - t_start) / n_run * remaining))
        print(f"  shard {i + 1}/{len(plan)} done in {dt:.0f}s "
              f"({len(chunk)} proteins) | {remaining} left, eta {eta}")

    n = merge_shards(p["out"], p["work"], plan, names)
    if not keep_shards:
        shutil.rmtree(p["work"], ignore_errors=True)
    print(f"[{label}] wrote {os.path.basename(p['out'])} — {n} x ({N_C},) "
          f"in {timedelta(seconds=int(time.time() - t_start))}")
    return True

## 5. Run the Embedding Pipeline

Process the splits identified by the inventory as requiring work. Valid completed splits are skipped, while invalid output files are moved aside before rebuilding.

This section performs the actual embedding computation. Run it only when you intend to execute embedding in the current notebook session.

In [7]:
if not ok:
    print("Preflight failed — resolve the items above before running.")
elif not todo:
    print("Nothing to do: all ten splits are already embedded.")
else:
    results = {}
    for r in todo:
        if r["status"] == "REBUILD":
            p = split_paths(r["fold"], r["split"])
            bad = p["out"] + ".invalid"
            print(f"[{r['fold']}/{r['split']}] existing file invalid ({r['note']}); "
                  f"moving to {os.path.basename(bad)}")
            os.replace(p["out"], bad)
        results[f"{r['fold']}/{r['split']}"] = embed_split(r["fold"], r["split"])
        print()

    failed = [k for k, v in results.items() if not v]
    print(f"{sum(results.values())}/{len(results)} splits completed"
          + (f" | incomplete: {failed}" if failed else ""))

[d1/train] 1042 proteins, 9 shards of <=128, device -1


  shard 1/9 done in 219s (128 proteins) | 8 left, eta 0:29:15
  shard 2/9 done in 220s (128 proteins) | 7 left, eta 0:25:37
  shard 3/9 done in 217s (128 proteins) | 6 left, eta 0:21:52
  shard 4/9 done in 159s (128 proteins) | 5 left, eta 0:16:58
  shard 5/9 done in 93s (128 proteins) | 4 left, eta 0:12:06
  shard 6/9 done in 89s (128 proteins) | 3 left, eta 0:08:18
  shard 7/9 done in 92s (128 proteins) | 2 left, eta 0:05:11
  shard 8/9 done in 89s (128 proteins) | 1 left, eta 0:02:27
  shard 9/9 done in 15s (18 proteins) | 0 left, eta 0:00:00
[d1/train] wrote d1_tr_unique_embeddings_bepler.h5 — 1042 x (6165,) in 0:19:53

[d1/test] 104 proteins, 1 shards of <=128, device -1
  shard 1/1 done in 93s (104 proteins) | 0 left, eta 0:00:00
[d1/test] wrote d1_te_unique_embeddings_bepler.h5 — 104 x (6165,) in 0:01:33

2/2 splits completed


## 6. Verify Generated Embeddings

Recheck the embedding inventory after processing. Confirm that every selected split is marked `DONE` and that each protein has a valid, finite 6,165-dimensional embedding.

These files are the protein-conditioning inputs required by the CTDDG fine-tuning notebook.

In [8]:
rows = inventory()
todo = show_inventory(rows)

if not todo:
    print(
        "\n[OK] every split holds one finite (6165,) vector per protein — "
        "02_finetune_ctddg.ipynb will load these without complaint."
    )

fold  split   proteins  embedded  shards  status   note
-------------------------------------------------------
d1    train       1042      1042          DONE     
d1    test         104       104          DONE     

2 splits complete, 0 to do, 0 proteins remaining

[OK] every split holds one finite (6165,) vector per protein — 02_finetune_ctddg.ipynb will load these without complaint.
